### LangChain

In [35]:
# %pip install -U langchain langchain-openai

from pathlib import Path
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from rich import print
from pydantic import SecretStr


In [ ]:
def print_response(response: AIMessage):
    print(f'Response id: {response.id}')
    if response.usage_metadata is not None:
        input_tokens = response.usage_metadata.get('input_tokens', 0)
        cached_tokens = response.usage_metadata.get('input_token_details', {}).get('cache_read', 0)
        output_tokens = response.usage_metadata.get('output_tokens', 0)
        reasoning_tokens = response.usage_metadata.get('output_token_details', {}).get('reasoning', 0)

        print(f'Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)')

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)

In [ ]:
openai_api_key = SecretStr(Path('openai-secret-key-ai-integrations-developers.txt').read_text(encoding='utf-8').strip())
openai_model = ChatOpenAI(model_name='gpt-5-nano', openai_api_key=openai_api_key, reasoning_effort="low",)

In [23]:
messages = [
SystemMessage(content="You are a helpful assistant that translates English to French."),
HumanMessage(content="Translate the following English text to French: 'Hello, how are you?'")
]

In [25]:
response = openai_model.invoke(
    input= messages
)

In [28]:
print_response(response)

Response id: lc_run--01a07674-aabf-7050-aada-f00505f9c2a3-0

Input tokens: 36 (0 cached); Output tokens: 79 (64 reasoning)

-------------------- [Output] --------------------

Bonjour, comment ça va ?

In [30]:
from pydantic import BaseModel

class WorkshopBrief(BaseModel): 
    title:str
    audience: str
    duration_minutes:float
    key_takeaways: list[str]

openai_astructured_output_model = openai_model.with_structured_output(WorkshopBrief)

In [33]:
response = openai_astructured_output_model.invoke(
    input=[
        SystemMessage("You are en expert event organizer"), 
        HumanMessage("Design a beginner-friendly Saturday workshop about balcony herb gardening")
    ]
)
print(response)

WorkshopBrief(
    title='Balcony Herb Garden: A Beginner’s Saturday Workshop',
    audience='Beginners, apartment dwellers, and urban gardeners curious about growing culinary herbs on a 
balcony',
    duration_minutes=180.0,
    key_takeaways=[
        'Understand the essentials of balcony microclimates (sun exposure, wind, shade) and how they affect herb 
choices',
        'Choose the right containers, potting mix, and drainage for balcony spaces',
        'Select beginner-friendly herbs (e.g., basil, mint, parsley, chives, cilantro, thyme) and companion 
planting basics',
        'Create an efficient watering routine and simple soil care plan to prevent common issues like overwatering 
and nutrient deficiency',
        'Learn how to establish a small, low-maintenance herb garden layout for a 4–6 plant balcony setup',
        'Understand light, soil, and water needs to maximize herb flavor and growth in limited space',
        'Hands-on planting activity: potting up a starter herb tray and labeling',
        'Troubleshooting: identifying and managing common pests and diseases in container herbs',
        'Seasonal planning and maintenance calendar for year-round balcony herb growing',
        'Simple harvest and rejuvenation techniques to keep herbs productive and flavorful'
    ]
)

### Tools

In [34]:
from langchain_core.tools import tool

In [ ]:
@tool
def get_database_status(): 
    """Returns information about the current dataase status"""
    return "healthy"

In [ ]:
# Give the chat model permission to request this Python function as a tool.
openai_model_with_tools = openai_model.bind_tools([get_database_status])

# Start a new conversation. Keeping this list fresh prevents reuse of old tool-call IDs.
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Check the database status."),
]

# Ask the model for its next step. It can either answer normally or request one or more tools.
ai_message = openai_model_with_tools.invoke(messages)
messages.append(ai_message)

# Run every tool the model requested.
for tool_call in ai_message.tool_calls:
    # Pass the arguments selected by the model to the matching Python tool.
    tool_result = get_database_status.invoke(tool_call["args"])

    # Send the result back using the exact ID from this assistant message.
    # This ID links the result to the tool request and avoids an invalid-request error.
    messages.append(
        ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
    )

# The model now reads the tool result and writes its final user-facing answer.
final_response = openai_model_with_tools.invoke(messages)
print_response(final_response)


In [ ]:
from typing import List
from langchain_core.messages import BaseMessage

# Maps each tool name requested by the model to its Python implementation.
tools_registry = {get_database_status.name: get_database_status}


def run_agent_loop(conversation: List[BaseMessage]):
    MAX_ITERATIONS = 100
    finished_successfully = False

    for _ in range(MAX_ITERATIONS):
        # The model either produces a final answer or requests one or more tools.
        reply = openai_model_with_tools.invoke(conversation)
        conversation.append(reply)

        # No tool calls means the model has finished the conversation.
        if not reply.tool_calls:
            finished_successfully = True
            break

        # Run every requested tool and add its result to the conversation.
        for tool_call in reply.tool_calls:
            tool_call_id = tool_call["id"]
            tool_call_name = tool_call["name"]
            tool_call_args = tool_call["args"]

            result = tools_registry[tool_call_name].invoke(tool_call_args)
            conversation.append(
                ToolMessage(content=str(result), tool_call_id=tool_call_id)
            )

    # Stop a runaway loop instead of repeatedly calling the API forever.
    if not finished_successfully:
        raise RuntimeError(
            f"Could not finish the interaction within {MAX_ITERATIONS} iterations."
        )

    return conversation


In [ ]:
def print_conversation(conversation: List[BaseMessage]):
    """Print messages, including every LLM tool decision and tool result."""
    for index, message in enumerate(conversation, start=1):
        print(f"\n[bold cyan]Message {index} — {message.type.upper()}[/bold cyan]")

        # AI messages may contain normal text, tool calls, or both.
        if isinstance(message, AIMessage) and message.tool_calls:
            print("[bold yellow]The LLM decided to call tool(s):[/bold yellow]")
            for tool_call in message.tool_calls:
                print(f"  [yellow]Tool:[/yellow] {tool_call['name']}")
                print(f"  [yellow]Arguments:[/yellow] {tool_call['args']}")
                print(f"  [dim]Tool-call ID: {tool_call['id']}[/dim]")

            if message.content:
                print(f"[bold]LLM text:[/bold] {message.content}")

        # A ToolMessage is the Python tool's result sent back to the LLM.
        elif isinstance(message, ToolMessage):
            print(f"[bold green]Tool result:[/bold green] {message.content}")
            print(f"[dim]For tool-call ID: {message.tool_call_id}[/dim]")

        # System and human messages, plus final plain LLM answers, show their content.
        else:
            print(message.content)


In [78]:
from datetime import datetime
from zoneinfo import ZoneInfo


@tool
def get_current_time(timezone: str) -> str:
    """Return the current date and time for an IANA timezone, for example 'Europe/Sofia'."""
    # ZoneInfo converts the supplied timezone name into the requested local time.
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


@tool
def multiply_numbers(first_number: float, second_number: float) -> float:
    """Multiply two numbers and return the result."""
    # This is a local Python calculation; it does not call an external service.
    return first_number * second_number


In [79]:
# Put all tools in one list so they can be passed to the model together.
learning_tools = [get_database_status, get_current_time, multiply_numbers]

# The registry lets the agent loop find and run a requested tool by its name.
tools_registry = {tool.name: tool for tool in learning_tools}

# 'auto' lets the model decide whether to answer directly or request a suitable tool.
# It does not force a tool call on every prompt.
openai_model_with_tools = openai_model.bind_tools(
    learning_tools,
    tool_choice="auto",
)


In [ ]:
# Example 1: ask a question that requires the database-status tool.
# Run the cells that define the tools and run_agent_loop before running this cell.
database_conversation = [
    SystemMessage(content="You are a helpful operations assistant. Use tools when needed."),
    HumanMessage(content="Use the database-status tool and tell me whether the database is healthy."),
]

# The loop asks the model, runs any tool calls, and asks the model for a final answer.
run_agent_loop(database_conversation)
print_conversation(database_conversation)

# Example 2: the model can make more than one tool call before it answers.
time_and_math_conversation = [
    SystemMessage(content="You are a helpful assistant. Use the requested tools."),
    HumanMessage(
        content=(
            "Use get_current_time for Europe/Sofia and multiply_numbers for 12 and 7. "
            "Then summarize both results."
        )
    ),
]

run_agent_loop(time_and_math_conversation)
print_conversation(time_and_math_conversation)

# Example 3: with tool_choice='auto', the model can answer without calling a tool.
normal_conversation = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Explain what an API is in one sentence."),
]

run_agent_loop(normal_conversation)
print_conversation(normal_conversation)


In [ ]:
# Run once in the notebook kernel. These packages power the two retrievers below.
%pip install -U wikipedia arxiv


In [ ]:
from langchain_community.retrievers import ArxivRetriever, WikipediaRetriever

# WikipediaRetriever searches Wikipedia articles and returns LangChain Document objects.
wikipedia_retriever = WikipediaRetriever(
    top_k_results=2,
    doc_content_chars_max=1_500,
)

# ArxivRetriever searches research-paper abstracts from arXiv.
arxiv_retriever = ArxivRetriever(
    load_max_docs=2,
    get_full_documents=False,  # False returns abstracts instead of downloading full PDFs.
)


def print_documents(documents):
    """Print a compact, readable preview of retrieved LangChain Documents."""
    for number, document in enumerate(documents, start=1):
        title = document.metadata.get("title", "Untitled document")
        print(f"\n[bold cyan]Document {number}: {title}[/bold cyan]")
        print(document.page_content[:600])


In [ ]:
# Example 1: retrieve general background information from Wikipedia.
wikipedia_documents = wikipedia_retriever.invoke("retrieval-augmented generation")
print_documents(wikipedia_documents)


In [ ]:
# Example 2: retrieve recent research-paper abstracts from arXiv.
arxiv_documents = arxiv_retriever.invoke("large language model agents")
print_documents(arxiv_documents)


In [ ]:
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for a topic and return relevant article excerpts."""
    # The retriever returns Document objects. Limit each excerpt to keep the tool result concise.
    documents = wikipedia_retriever.invoke(query)

    if not documents:
        return "No Wikipedia results were found."

    return "\n\n".join(
        f"Title: {document.metadata.get('title', 'Untitled')}\n"
        f"Excerpt: {document.page_content[:1_000]}"
        for document in documents
    )


# Add the retrieval tool to the existing local tools.
learning_tools = [
    get_database_status,
    get_current_time,
    multiply_numbers,
    search_wikipedia,
]

# Rebuild both objects so the agent loop can find and call the new tool.
tools_registry = {tool.name: tool for tool in learning_tools}
openai_model_with_tools = openai_model.bind_tools(
    learning_tools,
    tool_choice="auto",
)


In [ ]:
# Example: the model decides to call search_wikipedia, then summarizes the retrieved excerpts.
wikipedia_tool_conversation = [
    SystemMessage(content="You are a research assistant. Use Wikipedia when current context is needed."),
    HumanMessage(content="Use Wikipedia to explain retrieval-augmented generation in two sentences."),
]

run_agent_loop(wikipedia_tool_conversation)
print_conversation(wikipedia_tool_conversation)
